
# Extended MIP stress test notebook

Deze notebook test veel meer dan enkel een triviaal conflict.

## Wat wordt getest?

- Enkel conflictsegment
- Meerdere opeenvolgende segmenten
- Path continuity
- Station dwell constraints
- Passenger vs freight priority
- Dynamic priority strategy
- In-execution segmenten
- Warm starts
- Conflicten over meerdere segmenten
- Solver robustness
- Timeline-validatie
- Overlap-detectie
- Feasibility checks

Doel:
je MIP-model zo snel mogelijk “breken” zodat je verborgen bugs vindt.


In [21]:
# Imports
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd

from domain.train import Train, TrainType, TrainSubtype
from domain.segment import Segment, SegmentType
from domain.schedule import Timetable, ScheduledTimes

from simulation.state import SystemState

from model.instance import build_instance
from model.solver import solve



# Scenario 1 — Basisconflict

Twee treinen willen exact hetzelfde segment tegelijk gebruiken.


In [22]:

segments = {
    "A-B": Segment(
        id="A-B",
        seg_type=SegmentType.BETWEEN_STATION,
        source="A",
        target="B",
    ),
}

trains = {
    1: Train(
        train_no=1,
        train_type=TrainType.PASSENGER,
        train_subtype=TrainSubtype.IC,
        path=("A-B",),
        halt_indicators={},
        dynamics={"A-B": "0-0"},
    ),

    2: Train(
        train_no=2,
        train_type=TrainType.FREIGHT,
        train_subtype=TrainSubtype.FREIGHT,
        path=("A-B",),
        halt_indicators={},
        dynamics={"A-B": "0-0"},
    ),
}

tt_data = {
    (1, "A-B"): ScheduledTimes(
        entry_seconds=0,
        exit_seconds=120,
        running_time=120,
        dwell_time=None,
    ),

    (2, "A-B"): ScheduledTimes(
        entry_seconds=0,
        exit_seconds=120,
        running_time=120,
        dwell_time=None,
    ),
}

timetable = Timetable(tt_data)

state = SystemState(
    trains=trains,
    timetable=timetable,
    start_time=0.0,
)

instance = build_instance(
    state=state,
    timetable=timetable,
    trains=trains,
    segments=segments,
    current_time=0.0,
    priority_strategy="static",
    weight_passenger=2,
    weight_freight=1,
    upgrade_weight=0,
    gamma=300,
)

solution = solve(
    instance,
    priority_strategy="static",
    verbose=False,
)

solution


[t=0] T=2 S=1 conf=1 vars=7 constr=6 bin=1


Solution(status=optimal, objective=120.00, runtime=0.00s)

In [23]:

print("STATUS:", solution.status)

print("\nARRIVALS")
print(solution.entry)

print("\nDEPARTURES")
print(solution.exit)

print("\nDELAYS")
print(solution.delay)



STATUS: optimal

ARRIVALS
{(1, 'A-B'): 0.0, (2, 'A-B'): 120.0}

DEPARTURES
{(1, 'A-B'): 120.0, (2, 'A-B'): 240.0}

DELAYS
{(1, 'A-B'): 0.0, (2, 'A-B'): 120.0}



# Scenario 2 — Multi-segment conflict corridor

Nu gebruiken beide treinen:

A-B -> B-C -> C-D

Hier test je:
- continuity constraints;
- cascading delays;
- ordering consistency.


In [24]:

segments = {
    "A-B": Segment("A-B", SegmentType.BETWEEN_STATION, "A", "B"),
    "B-C": Segment("B-C", SegmentType.BETWEEN_STATION, "B", "C"),
    "C-D": Segment("C-D", SegmentType.BETWEEN_STATION, "C", "D"),
}

trains = {
    10: Train(
        train_no=10,
        train_type=TrainType.PASSENGER,
        train_subtype=TrainSubtype.IC,
        path=("A-B", "B-C", "C-D"),
        halt_indicators={},
        dynamics={
            "A-B": "0-0",
            "B-C": "0-0",
            "C-D": "0-0",
        },
    ),

    20: Train(
        train_no=20,
        train_type=TrainType.FREIGHT,
        train_subtype=TrainSubtype.FREIGHT,
        path=("A-B", "B-C", "C-D"),
        halt_indicators={},
        dynamics={
            "A-B": "0-0",
            "B-C": "0-0",
            "C-D": "0-0",
        },
    ),
}

tt_data = {}

for t in [10, 20]:

    tt_data[(t, "A-B")] = ScheduledTimes(
        entry_seconds=0,
        exit_seconds=100,
        running_time=100,
        dwell_time=None,
    )

    tt_data[(t, "B-C")] = ScheduledTimes(
        entry_seconds=100,
        exit_seconds=220,
        running_time=120,
        dwell_time=None,
    )

    tt_data[(t, "C-D")] = ScheduledTimes(
        entry_seconds=220,
        exit_seconds=340,
        running_time=120,
        dwell_time=None,
    )

timetable = Timetable(tt_data)

state = SystemState(
    trains=trains,
    timetable=timetable,
)

instance = build_instance(
    state=state,
    timetable=timetable,
    trains=trains,
    segments=segments,
    current_time=0,
    priority_strategy="static",
    weight_passenger=3,
    weight_freight=1,
    upgrade_weight=0,
    gamma=300,
)

solution = solve(instance, verbose=False)

solution


[t=0] T=2 S=3 conf=3 vars=21 constr=22 bin=3


Solution(status=optimal, objective=120.00, runtime=0.00s)

In [25]:

rows = []

for (t, s), arr in solution.entry.items():

    rows.append({
        "train": t,
        "segment": s,
        "entry": arr,
        "exit": solution.exit[(t, s)],
        "delay": solution.delay[(t, s)],
    })

df = pd.DataFrame(rows).sort_values(["train", "entry"])
df


,train,segment,entry,exit,delay
0,10,A-B,0.0,100.0,0.0
1,10,B-C,100.0,220.0,0.0
2,10,C-D,220.0,340.0,0.0
3,20,A-B,120.0,220.0,120.0
4,20,B-C,220.0,340.0,120.0
5,20,C-D,340.0,460.0,120.0



# Scenario 3 — Station dwell test

Hier testen we:
- minimum dwell;
- no early departure;
- station conflicts.


In [26]:

segments = {
    "STATION": Segment(
        "STATION",
        SegmentType.STATION,
        "X",
        "X",
    ),
}

trains = {
    100: Train(
        train_no=100,
        train_type=TrainType.PASSENGER,
        train_subtype=TrainSubtype.IC,
        path=("STATION",),
        halt_indicators={"STATION": True},
        dynamics={},
    ),
}

tt_data = {
    (100, "STATION"): ScheduledTimes(
        entry_seconds=0,
        exit_seconds=180,
        running_time=None,
        dwell_time=180,
    )
}

timetable = Timetable(tt_data)

state = SystemState(
    trains=trains,
    timetable=timetable,
)

instance = build_instance(
    state=state,
    timetable=timetable,
    trains=trains,
    segments=segments,
    current_time=0,
    priority_strategy="static",
    weight_passenger=1,
    weight_freight=1,
    upgrade_weight=0,
    gamma=300,
)

solution = solve(instance, verbose=False)

solution


[t=0] T=1 S=1 conf=0 vars=3 constr=2 bin=0


Solution(status=optimal, objective=0.00, runtime=0.00s)


# Scenario 4 — In-execution train

Belangrijkste realistische test.

We simuleren:
- trein zit AL in segment;
- MIP mag hem niet teleporteren;
- entry moet gefixeerd blijven.


In [27]:

segments = {
    "A-B": Segment("A-B", SegmentType.BETWEEN_STATION, "A", "B"),
}

trains = {
    1: Train(
        train_no=1,
        train_type=TrainType.PASSENGER,
        train_subtype=TrainSubtype.IC,
        path=("A-B",),
        halt_indicators={},
        dynamics={"A-B": "0-0"},
    )
}

tt_data = {
    (1, "A-B"): ScheduledTimes(
        entry_seconds=0,
        exit_seconds=300,
        running_time=300,
        dwell_time=None,
    )
}

timetable = Timetable(tt_data)

state = SystemState(
    trains=trains,
    timetable=timetable,
)

# Trein reeds gestart
state.record_entry(
    train_id=1,
    segment_id="A-B",
    time=50,
)

instance = build_instance(
    state=state,
    timetable=timetable,
    trains=trains,
    segments=segments,
    current_time=100,
    priority_strategy="static",
    weight_passenger=1,
    weight_freight=1,
    upgrade_weight=0,
    gamma=300,
)

solution = solve(instance, verbose=False)

solution


[t=100] T=1 S=1 conf=0 vars=3 constr=2 bin=0


Solution(status=optimal, objective=50.00, runtime=0.00s)

In [28]:

print(solution.entry)
print(solution.exit)


{(1, 'A-B'): 50.0}
{(1, 'A-B'): 350.0}



# Scenario 5 — Dynamic priorities

Test:
- vertraagde passenger krijgt hogere prioriteit;
- freight zou moeten wijken.


In [29]:

segments = {
    "A-B": Segment("A-B", SegmentType.BETWEEN_STATION, "A", "B"),
}

trains = {
    1: Train(
        train_no=1,
        train_type=TrainType.PASSENGER,
        train_subtype=TrainSubtype.IC,
        path=("A-B",),
        halt_indicators={},
        dynamics={"A-B": "0-0"},
    ),

    2: Train(
        train_no=2,
        train_type=TrainType.FREIGHT,
        train_subtype=TrainSubtype.FREIGHT,
        path=("A-B",),
        halt_indicators={},
        dynamics={"A-B": "0-0"},
    ),
}

tt_data = {
    (1, "A-B"): ScheduledTimes(
        entry_seconds=0,
        exit_seconds=100,
        running_time=100,
        dwell_time=None,
    ),

    (2, "A-B"): ScheduledTimes(
        entry_seconds=0,
        exit_seconds=100,
        running_time=100,
        dwell_time=None,
    ),
}

timetable = Timetable(tt_data)

state = SystemState(
    trains=trains,
    timetable=timetable,
)

# Passenger heeft al vertraging opgelopen
state.record_entry(1, "A-B", 0)
state.record_exit(1, "A-B", 500)

instance = build_instance(
    state=state,
    timetable=timetable,
    trains=trains,
    segments=segments,
    current_time=500,
    priority_strategy="dynamic",
    weight_passenger=2,
    weight_freight=1,
    upgrade_weight=100,
    gamma=60,
)

solution = solve(
    instance,
    priority_strategy="dynamic",
    verbose=False,
)

solution


[t=500] T=1 S=1 conf=0 vars=3 constr=2 bin=0


Solution(status=optimal, objective=500.00, runtime=0.00s)


# Feasibility validator

Detecteert:
- overlapping occupancies;
- continuity fouten;
- negative durations.


In [30]:

def validate_solution(solution):

    print("=" * 60)
    print("VALIDATION")
    print("=" * 60)

    # negatieve duur
    for key in solution.entry:

        arr = solution.entry[key]
        dep = solution.exit[key]

        if dep < arr:
            print("NEGATIVE DURATION:", key)

    # overlap per segment
    seg_usage = {}

    for (t, s), arr in solution.entry.items():

        dep = solution.exit[(t, s)]

        seg_usage.setdefault(s, []).append(
            (arr, dep, t)
        )

    for seg, usages in seg_usage.items():

        usages = sorted(usages)

        for i in range(len(usages) - 1):

            a1, d1, t1 = usages[i]
            a2, d2, t2 = usages[i + 1]

            if a2 < d1:
                print(f"OVERLAP on {seg}: train {t1} and {t2}")

    print("Validation finished.")


In [31]:
validate_solution(solution)


VALIDATION
Validation finished.



# Stress test — many trains

Hier zie je:
- solve time scaling;
- ordering explosion;
- potential symmetry issues.


In [32]:

N = 8

segments = {
    "MAIN": Segment(
        "MAIN",
        SegmentType.BETWEEN_STATION,
        "A",
        "B",
    )
}

trains = {}

tt_data = {}

for i in range(N):

    train_type = (
        TrainType.PASSENGER
        if i % 2 == 0
        else TrainType.FREIGHT
    )

    subtype = (
        TrainSubtype.IC
        if i % 2 == 0
        else TrainSubtype.FREIGHT
    )

    trains[i] = Train(
        train_no=i,
        train_type=train_type,
        train_subtype=subtype,
        path=("MAIN",),
        halt_indicators={},
        dynamics={"MAIN": "0-0"},
    )

    tt_data[(i, "MAIN")] = ScheduledTimes(
        entry_seconds=0,
        exit_seconds=100,
        running_time=100,
        dwell_time=None,
    )

timetable = Timetable(tt_data)

state = SystemState(
    trains=trains,
    timetable=timetable,
)

instance = build_instance(
    state=state,
    timetable=timetable,
    trains=trains,
    segments=segments,
    current_time=0,
    priority_strategy="static",
    weight_passenger=2,
    weight_freight=1,
    upgrade_weight=0,
    gamma=300,
)

solution = solve(
    instance,
    verbose=False,
)

print(solution.status)
print(solution.objective)
print("n entry vars:", len(solution.entry))


[t=0] T=8 S=1 conf=28 vars=52 constr=72 bin=28
optimal
3400.0
n entry vars: 8


In [36]:
# ============================================================
# Scenario — REAL dynamic priority test
#
# T1 = vertraagde passenger, nog actief
# T2 = freight
#
# Conflict ontstaat op toekomstig segment B-C.
#
# Verwachting:
# dynamic priority geeft T1 voorrang op B-C
# ondanks dat beide tegelijk willen binnenkomen.
# ============================================================

segments = {
    "A-B": Segment("A-B", SegmentType.BETWEEN_STATION, "A", "B"),
    "B-C": Segment("B-C", SegmentType.BETWEEN_STATION, "B", "C"),
}

trains = {
    1: Train(
        train_no=1,
        train_type=TrainType.PASSENGER,
        train_subtype=TrainSubtype.IC,
        path=("A-B", "B-C"),
        halt_indicators={},
        dynamics={
            "A-B": "0-0",
            "B-C": "0-0",
        },
    ),

    2: Train(
        train_no=2,
        train_type=TrainType.FREIGHT,
        train_subtype=TrainSubtype.FREIGHT,
        path=("B-C",),
        halt_indicators={},
        dynamics={
            "B-C": "0-0",
        },
    ),
}

# ------------------------------------------------------------
# Timetable
#
# Passenger:
#   A-B : 0 -> 100
#   B-C : 100 -> 200
#
# Freight:
#   B-C : 100 -> 200
#
# Dus conflict op B-C.
# ------------------------------------------------------------

tt_data = {

    # Passenger
    (1, "A-B"): ScheduledTimes(
        entry_seconds=0,
        exit_seconds=100,
        running_time=100,
        dwell_time=None,
    ),

    (1, "B-C"): ScheduledTimes(
        entry_seconds=100,
        exit_seconds=200,
        running_time=100,
        dwell_time=None,
    ),

    # Freight
    (2, "B-C"): ScheduledTimes(
        entry_seconds=100,
        exit_seconds=200,
        running_time=100,
        dwell_time=None,
    ),
}

timetable = Timetable(tt_data)

# ============================================================
# State
# ============================================================

state = SystemState(
    trains=trains,
    timetable=timetable,
)

# ------------------------------------------------------------
# Passenger is AL actief op A-B
#
# Gepland:
#   exit A-B = 100
#
# Werkelijk:
#   nog bezig op t=400
#
# => enorme vertraging
# ------------------------------------------------------------

state.record_entry(
    train_id=1,
    segment_id="A-B",
    time=0,
)

# NIET record_exit !!!
# Trein moet actief blijven.

# ============================================================
# Build instance
# ============================================================

instance = build_instance(
    state=state,
    timetable=timetable,
    trains=trains,
    segments=segments,

    current_time=400,

    priority_strategy="dynamic",

    weight_passenger=2,
    weight_freight=1,

    upgrade_weight=100,
    gamma=60,
)

print("=" * 60)
print("INSTANCE")
print("=" * 60)

print("T =", instance["T"])
print("S =", instance["S"])

print("\nWeights:")
print(instance["weights"])

print("\nConflicts:")
for s, pairs in instance["conflicts"].items():
    print(s, pairs)

# ============================================================
# Solve
# ============================================================

solution = solve(
    instance,
    priority_strategy="dynamic",
    verbose=False,
)

print("\n" + "=" * 60)
print("SOLUTION")
print("=" * 60)

print(solution)

print("\nARRIVALS")
for k, v in solution.entry.items():
    print(k, v)

print("\nDEPARTURES")
for k, v in solution.exit.items():
    print(k, v)

print("\nDELAYS")
for k, v in solution.delay.items():
    print(k, v)


# ============================================================
# Analyse
# ============================================================

rows = []

for (t, s), arr in solution.entry.items():

    rows.append({
        "train": t,
        "segment": s,
        "arrival": arr,
        "departure": solution.exit[(t, s)],
        "delay": solution.delay[(t, s)],
    })

df = pd.DataFrame(rows).sort_values(["arrival", "train"])

print("\n")
display(df)

print("\n" + "=" * 60)
print("EXPECTED")
print("=" * 60)

print("""
Trein 1 heeft gigantische vertraging en hoge dynamic priority.

Dus op B-C verwacht je normaal:

Passenger eerst:
    T1 enters B-C
    T2 waits

Niet:
    freight eerst
""")

INSTANCE
T = [1, 2]
S = {'A-B', 'B-C'}

Weights:
{1: 2, 2: 1}

Conflicts:
A-B []
B-C [(1, 2)]
[t=400] T=2 S=2 conf=1 vars=10 constr=9 bin=1

SOLUTION
Solution(status=optimal, objective=1000.00, runtime=0.00s)

ARRIVALS
(1, 'A-B') 0.0
(1, 'B-C') 400.0
(2, 'B-C') 500.0

DEPARTURES
(1, 'A-B') 400.0
(1, 'B-C') 500.0
(2, 'B-C') 600.0

DELAYS
(1, 'A-B') 0.0
(1, 'B-C') 300.0
(2, 'B-C') 400.0




,train,segment,arrival,departure,delay
0,1,A-B,0.0,400.0,0.0
1,1,B-C,400.0,500.0,300.0
2,2,B-C,500.0,600.0,400.0



EXPECTED

Trein 1 heeft gigantische vertraging en hoge dynamic priority.

Dus op B-C verwacht je normaal:

Passenger eerst:
    T1 enters B-C
    T2 waits

Niet:
    freight eerst



In [37]:
# ============================================================
# RIGOROUS in-execution test
#
# Doel:
# bewijzen of occupied / remaining runtime correct werkt.
#
# We testen hetzelfde actieve segment op meerdere current_time's.
#
# Als occupied correct is:
#
# current_time=100  -> exit=350
# current_time=200  -> exit=350
# current_time=300  -> exit=350
#
# Dus:
# FIXED fysieke exit
#
# Als occupied fout is:
#
# current_time=100  -> 350
# current_time=200  -> 450
# current_time=300  -> 550
#
# Dan wordt volledige runtime telkens opnieuw toegevoegd.
# ============================================================

segments = {
    "A-B": Segment(
        "A-B",
        SegmentType.BETWEEN_STATION,
        "A",
        "B",
    ),
}

trains = {
    1: Train(
        train_no=1,
        train_type=TrainType.PASSENGER,
        train_subtype=TrainSubtype.IC,
        path=("A-B",),
        halt_indicators={},
        dynamics={"A-B": "0-0"},
    )
}

# Runtime = 300
# Actual entry = 50
#
# Dus fysieke exit MOET altijd:
#
# 50 + 300 = 350
#
# onafhankelijk van current_time.

tt_data = {
    (1, "A-B"): ScheduledTimes(
        entry_seconds=0,
        exit_seconds=300,
        running_time=300,
        dwell_time=None,
    )
}

timetable = Timetable(tt_data)

# ============================================================
# State
# ============================================================

state = SystemState(
    trains=trains,
    timetable=timetable,
)

# Trein start werkelijk op t=50
state.record_entry(
    train_id=1,
    segment_id="A-B",
    time=50,
)

# ============================================================
# Sweep over meerdere current_time's
# ============================================================

results = []

for current_time in [60, 100, 150, 200, 250, 300]:

    print("\n" + "=" * 70)
    print(f"CURRENT TIME = {current_time}")
    print("=" * 70)

    instance = build_instance(
        state=state,
        timetable=timetable,
        trains=trains,
        segments=segments,

        current_time=current_time,

        priority_strategy="static",

        weight_passenger=1,
        weight_freight=1,

        upgrade_weight=0,
        gamma=300,
    )

    # --------------------------------------------------------
    # Inspecteer instance
    # --------------------------------------------------------

    print("\nfixed_entry:")
    print(instance.get("fixed_entry", {}))

    print("\noccupied:")
    print(instance.get("occupied", {}))

    print("\nruntime:")
    print(instance.get("runtime", {}))

    # --------------------------------------------------------
    # Solve
    # --------------------------------------------------------

    solution = solve(
        instance,
        priority_strategy="static",
        verbose=False,
    )

    entry = solution.entry[(1, "A-B")]
    exit_ = solution.exit[(1, "A-B")]

    duration = exit_ - entry

    results.append({
        "current_time": current_time,
        "entry": entry,
        "exit": exit_,
        "duration": duration,
    })

    print("\nsolution.entry")
    print(solution.entry)

    print("\nsolution.exit")
    print(solution.exit)

    print("\nduration")
    print(duration)

# ============================================================
# Analyse
# ============================================================

print("\n" + "=" * 70)
print("FINAL ANALYSIS")
print("=" * 70)

df = pd.DataFrame(results)

display(df)

print("""
EXPECTED BEHAVIOUR:

entry should ALWAYS stay 50
exit should ALWAYS stay 350

because:
- train physically entered at t=50
- runtime is fixed at 300
- physical exit must therefore be 350

If exit increases together with current_time,
then occupied / remaining runtime logic is broken.
""")


CURRENT TIME = 60

fixed_entry:
{(1, 'A-B'): 50}

occupied:
{(1, 'A-B'): 290}

runtime:
{(1, 'A-B'): 300}
[t=60] T=1 S=1 conf=0 vars=3 constr=2 bin=0

solution.entry
{(1, 'A-B'): 50.0}

solution.exit
{(1, 'A-B'): 350.0}

duration
300.0

CURRENT TIME = 100

fixed_entry:
{(1, 'A-B'): 50}

occupied:
{(1, 'A-B'): 250}

runtime:
{(1, 'A-B'): 300}
[t=100] T=1 S=1 conf=0 vars=3 constr=2 bin=0

solution.entry
{(1, 'A-B'): 50.0}

solution.exit
{(1, 'A-B'): 350.0}

duration
300.0

CURRENT TIME = 150

fixed_entry:
{(1, 'A-B'): 50}

occupied:
{(1, 'A-B'): 200}

runtime:
{(1, 'A-B'): 300}
[t=150] T=1 S=1 conf=0 vars=3 constr=2 bin=0

solution.entry
{(1, 'A-B'): 50.0}

solution.exit
{(1, 'A-B'): 350.0}

duration
300.0

CURRENT TIME = 200

fixed_entry:
{(1, 'A-B'): 50}

occupied:
{(1, 'A-B'): 150}

runtime:
{(1, 'A-B'): 300}
[t=200] T=1 S=1 conf=0 vars=3 constr=2 bin=0

solution.entry
{(1, 'A-B'): 50.0}

solution.exit
{(1, 'A-B'): 350.0}

duration
300.0

CURRENT TIME = 250

fixed_entry:
{(1, 'A-B'):

,current_time,entry,exit,duration
0,60,50.0,350.0,300.0
1,100,50.0,350.0,300.0
2,150,50.0,350.0,300.0
3,200,50.0,350.0,300.0
4,250,50.0,350.0,300.0
5,300,50.0,350.0,300.0



EXPECTED BEHAVIOUR:

entry should ALWAYS stay 50
exit should ALWAYS stay 350

because:
- train physically entered at t=50
- runtime is fixed at 300
- physical exit must therefore be 350

If exit increases together with current_time,
then occupied / remaining runtime logic is broken.

